## Example 1: Client initialization and inspection

In [ ]:
from pystac_client import Client
from pystac_client.conformance import ConformanceClasses
import pystac_client

# Example 1: Client initialization and inspection
print("Testing pystac_client Client:")

# Show version
print(f"✓ pystac_client version: {pystac_client.__version__}")

# Demonstrate Client.open() method
api_url = "https://earth-search.aws.element84.com/v1"
print(f"✓ Opening STAC API at: {api_url}")

try:
    # Open a STAC API
    client = Client.open(api_url)
    print(f"✓ Client opened: {client.title}")
    print(f"✓ Client ID: {client.id}")
    print(f"✓ Description: {client.description[:50]}...")
    conformance = client.get_conforms_to()
    assert conformance, "an opened client must advertise at least one conformance class"
    print(f"✓ Conformance classes: {len(conformance)}")
    assert client.conforms_to(ConformanceClasses.CORE), (
        f"a STAC API must advertise the CORE conformance class; got {conformance}"
    )
    try:
        client.conforms_to("https://example.invalid/not-a-conformance-class")
    except ValueError:
        pass
    else:
        raise AssertionError(
            "conforms_to() accepted an unknown conformance class -- it is not reading its argument"
        )
    
    # List collections
    collections = list(client.get_collections())
    print(f"✓ Available collections: {len(collections)}")
    if collections:
        print(f"✓ First collection: {collections[0].id}")
except Exception as e:
    print("✗ pystac_client Client example did not complete")
    raise AssertionError(
        f"pystac_client example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Item search

In [ ]:
from pystac_client import Client

# Example 2: Item search
print("Testing pystac_client ItemSearch:")

api_url = "https://earth-search.aws.element84.com/v1"
_COLLECTION = "sentinel-2-l2a"
_BBOX = [-122.5, 37.5, -122.0, 38.0]

try:
    client = Client.open(api_url)

    # Create item search with parameters
    search = client.search(
        collections=[_COLLECTION],
        bbox=_BBOX,
        datetime="2023-06-01/2023-06-30",
        limit=5,
    )

    assert type(search).__name__ == "ItemSearch", f"client.search() returned {type(search).__name__}"
    # ItemSearch.url is the search endpoint (a str); url_with_parameters() adds the query string.
    assert isinstance(search.url, str), f"ItemSearch.url should be a str, got {type(search.url)}"
    assert search.url == f"{api_url}/search", f"search endpoint should be {api_url}/search, got {search.url!r}"
    _parameterised = search.url_with_parameters()
    for _fragment in ("limit=5", "bbox=", "datetime=", "collections="):
        assert _fragment in _parameterised, (
            f"url_with_parameters() dropped {_fragment!r} -- the search parameters never reached the "
            f"query string: {_parameterised}"
        )
    print(f"✓ Created search: {type(search).__name__}")
    print(f"✓ Search URL: {search.url}")
    print(f"✓ Parameterised: {_parameterised}")

    # Get items
    items = list(search.items())
    # An empty result would mean the search failed silently.
    assert items, (
        "the search returned no items; the query or the endpoint is not working"
    )
    print(f"✓ Found {len(items)} items")

    # Every item must satisfy the query we sent -- collection, bbox overlap and month. A server that
    # ignored the filters would return items and still fail here.
    for _item in items:
        assert _item.collection_id == _COLLECTION, (
            f"item {_item.id} is in collection {_item.collection_id!r}, not {_COLLECTION!r}"
        )
        assert _item.datetime.year == 2023 and _item.datetime.month == 6, (
            f"item {_item.id} is dated {_item.datetime}, outside the requested 2023-06 window"
        )
        _west, _south, _east, _north = _item.bbox
        assert _west <= _BBOX[2] and _east >= _BBOX[0] and _south <= _BBOX[3] and _north >= _BBOX[1], (
            f"item {_item.id} bbox {_item.bbox} does not overlap the requested {_BBOX}"
        )
        assert _item.assets, f"item {_item.id} carries no assets"

    item = items[0]
    print(f"✓ First item ID: {item.id}")
    print(f"✓ Item datetime: {item.datetime}")
    print(f"✓ Item assets: {list(item.assets.keys())[:3]}")
    print(f"✓ All {len(items)} items are in {_COLLECTION}, dated 2023-06, and overlap the bbox")
except Exception as e:
    print("✗ pystac_client ItemSearch example did not complete")
    raise AssertionError(
        f"pystac_client example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: Collection access and metadata

In [ ]:
from pystac_client import Client

# Example 3: Collection access and metadata
print("Testing pystac_client collection operations:")

api_url = "https://earth-search.aws.element84.com/v1"

try:
    client = Client.open(api_url)
    
    # Get specific collection
    collection = client.get_collection("sentinel-2-l2a")
    print(f"✓ Collection ID: {collection.id}")
    print(f"✓ Collection title: {collection.title}")
    print(f"✓ Collection license: {collection.license}")
    
    # Access extent
    extent = collection.extent
    bbox = extent.spatial.bboxes[0]
    print(f"✓ Spatial extent (bbox): {bbox}")
    
    temporal = extent.temporal.intervals[0]
    print(f"✓ Temporal extent: {temporal[0]} to {temporal[1]}")
    
    # Access summaries
    if collection.summaries:
        print(f"✓ Summaries: {list(collection.summaries.lists.keys())[:3]}")
except Exception as e:
    print("✗ pystac_client collection example did not complete")
    raise AssertionError(
        f"pystac_client example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: Search with a query filter

In [ ]:
from pystac_client import Client

# Example 4: Search with a query filter
print("Testing pystac_client advanced search:")

api_url = "https://earth-search.aws.element84.com/v1"
_COLLECTION = "sentinel-2-l2a"
_BBOX = [-122.5, 37.5, -122.0, 38.0]
_WINDOW = "2023-06-01/2023-06-30"
_CLOUD_COVER_LIMIT = 10

try:
    client = Client.open(api_url)

    # Search with query filter. `max_items` CAPS the iteration; `limit` only sets the page size.
    search = client.search(
        collections=[_COLLECTION],
        bbox=_BBOX,
        datetime=_WINDOW,
        query={"eo:cloud_cover": {"lt": _CLOUD_COVER_LIMIT}},
        limit=3,
        max_items=3,
    )
    # The same search without the filter, to show that the filter narrows the result.
    unfiltered = client.search(
        collections=[_COLLECTION], bbox=_BBOX, datetime=_WINDOW, limit=3, max_items=3
    )

    assert isinstance(search.url, str), f"ItemSearch.url should be a str, got {type(search.url)}"
    _parameterised = search.url_with_parameters()
    assert "query=" in _parameterised, (
        f"the eo:cloud_cover filter never reached the query string: {_parameterised}"
    )
    print(f"✓ Search with cloud cover < {_CLOUD_COVER_LIMIT}%")
    print(f"✓ Search URL: {search.url}")

    # Get matched count
    matched = search.matched()
    unfiltered_matched = unfiltered.matched()
    assert isinstance(matched, int) and matched >= 1, f"matched() must be a positive int, got {matched!r}"
    assert isinstance(unfiltered_matched, int), f"matched() must be an int, got {unfiltered_matched!r}"
    assert matched < unfiltered_matched, (
        f"the eo:cloud_cover filter did not narrow anything: it matched {matched} items and the "
        f"same search without `query=` matched {unfiltered_matched}"
    )
    print(f"✓ Matched items: {matched} (unfiltered: {unfiltered_matched})")

    # Iterate items — bounded by max_items above, and every one must satisfy the predicate we sent.
    _seen = 0
    for i, item in enumerate(search.items(), 1):
        cc = item.properties.get("eo:cloud_cover", "N/A")
        assert isinstance(cc, (int, float)), f"item {item.id} reports no numeric eo:cloud_cover ({cc!r})"
        assert cc < _CLOUD_COVER_LIMIT, (
            f"item {item.id} has eo:cloud_cover {cc}, which the `lt` {_CLOUD_COVER_LIMIT} filter "
            f"should have excluded — the server returned it anyway"
        )
        print(f"✓ Item {i}: {item.id}, cloud cover: {cc}%")
        _seen = i
    assert _seen >= 1, "the filtered search yielded no items to check the predicate against"
    assert _seen <= 3, f"max_items=3 did not cap the iteration; it yielded {_seen}"
except Exception as e:
    print("✗ pystac_client advanced-search example did not complete")
    raise AssertionError(
        f"pystac_client example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: Pagination and item collection

In [ ]:
from pystac_client import Client

# Example 5: Pagination and item collection
print("Testing pystac_client pagination:")

api_url = "https://earth-search.aws.element84.com/v1"

try:
    client = Client.open(api_url)
    
    # Create search
    search = client.search(
        collections=["sentinel-2-l2a"],
        bbox=[-122.5, 37.5, -122.0, 38.0],
        datetime="2023-06-01/2023-06-15",
        limit=10
    )
    
    # Get as item collection
    item_collection = search.item_collection()
    print(f"✓ Item collection type: {type(item_collection).__name__}")
    print(f"✓ Number of items: {len(item_collection)}")
    
    # Get items as dicts
    items_dict = search.items_as_dicts()
    print(f"✓ Items as dicts: {type(items_dict).__name__}")
    
    # Iterate with pagination
    count = 0
    for item in search.items():
        count += 1
        if count >= 5:
            break
    print(f"✓ Iterated {count} items with pagination")
except Exception as e:
    print("✗ pystac_client pagination example did not complete")
    raise AssertionError(
        f"pystac_client example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e